# ScriptSync - Video to Script (runs the full website in Colab)
Part of **NovaForge AI / Gpu4model supply**. Runs the same web app as `projects/scriptsync`, on Colab's GPU.

**Steps**
1. `Runtime -> Change runtime type -> T4 GPU`
2. `Runtime -> Run all`
3. Open the public link printed by the last cell. Keep this tab open while you use the tool.

The link changes every time you run it. No account or token is needed.

In [ ]:
import os, subprocess, sys
print(subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout or "No GPU found: Runtime > Change runtime type > T4 GPU")
os.chdir("/content")
subprocess.run("rm -rf Gpu4model-supply && git clone -q --depth 1 https://github.com/Ankita2005899/Gpu4model-supply.git", shell=True, check=True)
os.chdir("/content/Gpu4model-supply/projects/scriptsync")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "faster-whisper"], check=True)
print("Setup done:", os.getcwd())

In [ ]:
import os, subprocess, time, urllib.request
os.chdir("/content/Gpu4model-supply/projects/scriptsync")
env = dict(os.environ, PORT="5000", WHISPER_MODEL="medium", WHISPER_DEVICE="cuda", WHISPER_COMPUTE="float16")
# If you see a CUDA / cuDNN error in the log, change WHISPER_DEVICE to "cpu" and WHISPER_COMPUTE to "int8", then re-run this cell.
log = open("/content/scriptsync.log", "w")
server = subprocess.Popen([sys.executable, "app.py"], env=env, stdout=log, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:5000/api/ready", timeout=2)
        print("ScriptSync server is running.")
        break
    except Exception:
        time.sleep(2)
else:
    print("Server did not start. Log:")
    print(open("/content/scriptsync.log").read()[-3000:])

In [ ]:
import re, time
subprocess.run("wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared", shell=True, check=True)
tlog = open("/content/tunnel.log", "w")
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:5000", "--no-autoupdate"], stdout=tlog, stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        url = m.group(0)
        break
    time.sleep(1)
print("=" * 60)
print("Your ScriptSync website is live at:" if url else "Tunnel did not start. Re-run this cell.")
if url: print(url)
print("Keep this notebook open while you use it. The link changes every run.")
print("=" * 60)